In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import TransformedTargetRegressor
import torch.nn as nn
import torch.optim as optim
import xgboost as xgb
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
import matplotlib.pyplot as plt

from helper_modules import phase2_preprocess_data

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

# disable some warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="hyperopt")


In [ ]:
df = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_train.csv'))
print(df.shape)
display(df.head())

In [ ]:
# Try extracting data that has Demand_Response_Flag = 0 and Demand_Response_Capacity_kW != 0.0
df1 = df[
    (df['Demand_Response_Flag']==0) &
    (df['Demand_Response_Capacity_kW'] != 0.0)
].copy(deep=True)
print (df1.shape[0])

df1 = df[
    df['Demand_Response_Flag']==0
].copy(deep=True)
df2 = df1[
    df1['Demand_Response_Capacity_kW'] != 0.0
].copy(deep=True)
print (df2.shape[0])
# We observe that if Demand_Response_Flag = 0, then Demand_Response_Capacity_kW is always 0.0

df.groupby('Demand_Response_Flag')['Demand_Response_Capacity_kW'].agg(['min','max','mean','median','std','count'])

In [ ]:
# perform some preprocessing
df = phase2_preprocess_data(df)
display(df.head())

# Visualize the distribution of Demand_Response_Capacity_kW
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(9, 3))
df['Demand_Response_Capacity_kW'].hist(bins=100, ax=ax1)
ax1.set_title('Distribution of DRC(kW) w/ zeros')
ax1.set_xlabel('Demand_Response_Capacity_kW')
ax1.set_ylabel('Count')

df[
    df['Demand_Response_Capacity_kW'] != 0.0
]['Demand_Response_Capacity_kW'].hist(bins=100, ax=ax2)
ax2.set_title('Distribution of DRC(kW) w/out zeros')
ax2.set_xlabel('Demand_Response_Capacity_kW')
ax2.set_ylabel('Count')
plt.show()

In [ ]:
# Here we are gonna build 2 models:
# 1. Regression model to predict Demand_Response_Capacity_kW when Demand_Response_Flag = -1
# 2. Regression model to predict Demand_Response_Capacity_kW when Demand_Response_Flag = +1
# We know that when Demand_Response_Flag = 0, then Demand_Response_Capacity_kW is always 0.0

df = df.drop_duplicates()
df_neg = df[df['Demand_Response_Flag'] == -1].copy(deep=True)
df_pos = df[df['Demand_Response_Flag'] == 1].copy(deep=True)
print (df_neg.shape, df_pos.shape)

In [ ]:
best_hyperparams = {
    'reg_gamma': 1.7,
    'reg_learning_rate': 0.1,
    'reg_max_depth': 5,
    'reg_n_estimators': 800,
    'reg_reg_alpha': 2.0,
    'reg_reg_lambda': 3.0,
}

In [ ]:
# For negative Demand_Response_Flag
X = df_neg.drop(columns=['Demand_Response_Capacity_kW','Demand_Response_Flag']).values
y = df_neg['Demand_Response_Capacity_kW'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.02, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

reg_neg = xgb.XGBRegressor(
    objective='reg:squarederror',
    # objective='reg:pseudohubererror',
    n_estimators=int(best_hyperparams['reg_n_estimators']),
    max_depth=int(best_hyperparams['reg_max_depth']),
    learning_rate=best_hyperparams['reg_learning_rate'],
    gamma=best_hyperparams['reg_gamma'],
    reg_lambda=best_hyperparams['reg_reg_lambda'],
    reg_alpha=best_hyperparams['reg_reg_alpha'],
    random_state=42
).fit(X_train, y_train)

y_pred = reg_neg.predict(X_test)
print("Negative DRF Model Performance:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:",np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

# print some statistics on y_test and y_pred
print(y_test.min(), y_test.max(), y_test.mean(), y_test.std())
print(y_pred.min(), y_pred.max(), y_pred.mean(), y_pred.std())


In [ ]:
# For positive Demand_Response_Flag
X = df_pos.drop(columns=['Demand_Response_Capacity_kW','Demand_Response_Flag']).values
y = df_pos['Demand_Response_Capacity_kW'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

reg_pos = xgb.XGBRegressor(
    objective='reg:squarederror',
    # objective='reg:pseudohubererror',
    n_estimators=int(best_hyperparams['reg_n_estimators']),
    max_depth=int(best_hyperparams['reg_max_depth']),
    learning_rate=best_hyperparams['reg_learning_rate'],
    gamma=best_hyperparams['reg_gamma'],
    reg_lambda=best_hyperparams['reg_reg_lambda'],
    reg_alpha=best_hyperparams['reg_reg_alpha'],
    random_state=42
).fit(X_train, y_train)

y_pred = reg_pos.predict(X_test)
print("Positive DRF Model Performance:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:",np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

# print some statistics on y_test and y_pred
print(y_test.min(), y_test.max(), y_test.mean(), y_test.std())
print(y_pred.min(), y_pred.max(), y_pred.mean(), y_pred.std())


In [ ]:
# Save models
joblib.dump(reg_neg, './models/phase2_xgb_reg_neg_model.pkl')
joblib.dump(reg_pos, './models/phase2_xgb_reg_pos_model.pkl')
print("Models saved.")